In [ ]:
# ЯЧЕЙКА 1: Проверка качества интернета перед большими загрузками.
# ВАЖНО: по вашему ТЗ оставлена только интернет-диагностика.

import importlib.util  # Модуль для проверки, установлен ли пакет speedtest-cli.
import re  # Модуль для извлечения среднего пинга из текста команды ping.
import subprocess  # Модуль для запуска системных команд (pip, ping) из Python.
import sys  # Модуль для получения пути к текущему Python-интерпретатору.


def run_command_live(command_list, description=None):
    # Функция запускает внешнюю команду и печатает её вывод в реальном времени.
    # command_list: список аргументов команды, например ["ping", "-c", "3", "civitai.com"].
    # description: текстовое описание шага для читабельного лога.
    if description is not None:  # Проверяем, передано ли описание команды.
        print(f"[RUN] {description}")  # Печатаем название шага.
    print("[CMD]", " ".join(command_list))  # Печатаем итоговую команду целиком.

    process = subprocess.Popen(  # Создаём процесс с объединённым stdout/stderr.
        command_list,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines = []  # Список для накопления всех строк вывода команды.
    if process.stdout is not None:  # Защита от редкого случая отсутствия stdout.
        for output_line in process.stdout:  # Читаем вывод построчно.
            output_lines.append(output_line)  # Сохраняем строку в список.
            print(output_line, end="")  # Сразу отображаем строку в ноутбуке.

    exit_code = process.wait()  # Ждём завершения команды и получаем код выхода.
    return exit_code, "".join(output_lines)  # Возвращаем код и полный текст вывода.


def ensure_speedtest_cli_installed():
    # Функция проверяет наличие speedtest-cli и устанавливает его, если пакета нет.
    is_installed = importlib.util.find_spec("speedtest") is not None  # Флаг: найден ли модуль speedtest.
    if is_installed:  # Если модуль уже существует.
        print("speedtest-cli уже установлен.")
        return True  # Возвращаем успешный результат.

    print("speedtest-cli не найден. Устанавливаю пакет...")
    exit_code, _ = run_command_live(  # Запускаем установку через текущий Python.
        [sys.executable, "-m", "pip", "install", "-U", "speedtest-cli"],
        description="Установка speedtest-cli",
    )
    return exit_code == 0  # Возвращаем True, если pip завершился без ошибок.


def run_speedtest_mbps():
    # Функция выполняет тест скорости и возвращает словарь с результатами.
    speed_results = {  # Словарь для структурированного хранения измерений.
        "download_mbps": None,  # Скорость загрузки в Мбит/с.
        "upload_mbps": None,  # Скорость отдачи в Мбит/с.
        "error": None,  # Текст ошибки, если speedtest не отработал.
    }

    try:  # Блок попытки запуска speedtest.
        import speedtest  # Импорт библиотеки speedtest-cli.

        tester = speedtest.Speedtest(secure=True)  # Объект для выполнения теста скорости.
        tester.get_best_server()  # Выбираем ближайший оптимальный сервер теста.
        download_bps = tester.download()  # Получаем download в бит/сек.
        upload_bps = tester.upload()  # Получаем upload в бит/сек.

        speed_results["download_mbps"] = download_bps / 1_000_000  # Перевод в Мбит/с.
        speed_results["upload_mbps"] = upload_bps / 1_000_000  # Перевод в Мбит/с.
    except Exception as error_object:  # Ловим любую ошибку, чтобы не прерывать ноутбук.
        speed_results["error"] = str(error_object)  # Сохраняем текст ошибки.

    return speed_results  # Возвращаем итоговый словарь измерений.


def get_average_ping_ms(hostname):
    # Функция измеряет средний ping до указанного хоста.
    # hostname: домен назначения, например "civitai.com" или "huggingface.co".
    ping_command = ["ping", "-c", "3", "-W", "2", hostname]  # Команда Linux ping на 3 пакета.
    exit_code, command_output = run_command_live(ping_command, description=f"Ping до {hostname}")

    if exit_code != 0:  # Если ping завершился ошибкой.
        return None, f"ping завершился с кодом {exit_code}"  # Возвращаем ошибку.

    match_object = re.search(r"=\s*([\d\.]+)/([\d\.]+)/([\d\.]+)/([\d\.]+)", command_output)  # Парсим строку rtt.
    if match_object is None:  # Если формат вывода не распознан.
        return None, "не удалось распарсить средний ping"

    average_ping_ms = float(match_object.group(2))  # Берём второе число: avg ping.
    return average_ping_ms, None  # Возвращаем средний ping и отсутствие ошибки.


print("=== INTERNET CHECK ===")  # Заголовок секции проверки сети.
speedtest_ready = ensure_speedtest_cli_installed()  # Флаг готовности speedtest-cli.

network_info = {  # Словарь для сводного отчёта по интернету.
    "download_mbps": None,  # Итоговый download в Мбит/с.
    "upload_mbps": None,  # Итоговый upload в Мбит/с.
    "civitai_ping_ms": None,  # Средний ping до civitai.com.
    "hf_ping_ms": None,  # Средний ping до huggingface.co.
}

if speedtest_ready:  # Если speedtest-cli доступен.
    speed_data = run_speedtest_mbps()  # Выполняем замер скорости.
    network_info["download_mbps"] = speed_data["download_mbps"]  # Записываем download.
    network_info["upload_mbps"] = speed_data["upload_mbps"]  # Записываем upload.
    if speed_data["error"] is not None:  # Если speedtest вернул ошибку.
        print(f"[WARN] Ошибка speedtest: {speed_data['error']}")  # Предупреждение без остановки.
else:  # Если speedtest-cli не удалось установить.
    print("[WARN] speedtest-cli недоступен, проверка скорости пропущена.")

civitai_ping_ms, civitai_ping_error = get_average_ping_ms("civitai.com")  # Замер ping для Civitai.
hf_ping_ms, hf_ping_error = get_average_ping_ms("huggingface.co")  # Замер ping для HuggingFace.
network_info["civitai_ping_ms"] = civitai_ping_ms  # Сохраняем ping Civitai.
network_info["hf_ping_ms"] = hf_ping_ms  # Сохраняем ping HF.

print("
=== RESULT ===")
print(f"Download: {network_info['download_mbps']:.2f} Mbps" if network_info["download_mbps"] is not None else "Download: N/A")
print(f"Upload:   {network_info['upload_mbps']:.2f} Mbps" if network_info["upload_mbps"] is not None else "Upload: N/A")
print(f"Ping civitai.com: {network_info['civitai_ping_ms']:.2f} ms" if network_info["civitai_ping_ms"] is not None else f"Ping civitai.com: N/A ({civitai_ping_error})")
print(f"Ping huggingface.co: {network_info['hf_ping_ms']:.2f} ms" if network_info["hf_ping_ms"] is not None else f"Ping huggingface.co: N/A ({hf_ping_error})")

# КРИТЕРИИ по ТЗ:
# - скорость download должна быть выше 300 Мбит/с,
# - критерий ping оставляем прежним: средний ping < 100 мс для обоих хостов.
download_ok = network_info["download_mbps"] is not None and network_info["download_mbps"] > 300  # Проверка download > 300.
civitai_ping_ok = network_info["civitai_ping_ms"] is not None and network_info["civitai_ping_ms"] < 100  # Проверка ping Civitai.
hf_ping_ok = network_info["hf_ping_ms"] is not None and network_info["hf_ping_ms"] < 100  # Проверка ping HF.

if download_ok and civitai_ping_ok and hf_ping_ok:  # Все критерии выполнены.
    print("[OK] Интернет соответствует рекомендованному уровню для работы.")
else:  # Хотя бы один критерий не выполнен.
    print("[WARN] Интернет ниже рекомендуемого порога (download > 300 Mbps и ping < 100 ms).")


In [ ]:
# ЯЧЕЙКА 2: Главные настройки для Vast + ComfyUI.
# ВАЖНО: секреты (CIVITAI_TOKEN, HF_TOKEN, GRADIO_TOKEN и т.д.)
# должны быть заданы в окружении ДО запуска ноутбука.

import os  # Модуль для чтения переменных окружения и работы с путями.
from pathlib import Path  # Класс Path для удобной и безопасной работы с директориями.

# Базовая папка постоянного тома Vast.ai, где сохраняются данные между перезапусками.
VAST_VOLUME_DIR = Path("/workspace")
# Корневая папка проекта ComfyUI.
COMFY_ROOT = VAST_VOLUME_DIR / "ComfyUI"

# Папки моделей внутри ComfyUI (актуальные пути под новую платформу).
MODELS_DIR = COMFY_ROOT / "models"
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints"  # Папка для основных SD/SDXL моделей.
LORAS_DIR = MODELS_DIR / "loras"  # Папка для LoRA-моделей.
VAE_DIR = MODELS_DIR / "vae"  # Папка для VAE-файлов.
UPSCALE_DIR = MODELS_DIR / "upscale_models"  # Папка для апскейлеров (ESRGAN/4x и т.п.).
CONTROLNET_DIR = MODELS_DIR / "controlnet"  # Папка для ControlNet моделей.
IPADAPTER_DIR = MODELS_DIR / "ipadapter"  # Папка для IP-Adapter моделей.
CLIP_VISION_DIR = MODELS_DIR / "clip_vision"  # Папка для CLIP Vision моделей, нужных IP-Adapter.

# Папки расширений и служебных данных ComfyUI.
CUSTOM_NODES_DIR = COMFY_ROOT / "custom_nodes"  # Папка для кастомных нод (расширений).
INPUT_DIR = COMFY_ROOT / "input"  # Папка входных изображений для workflow.
OUTPUT_DIR = COMFY_ROOT / "output"  # Папка результатов генерации.
USER_DIR = COMFY_ROOT / "user"  # Папка пользовательских настроек ComfyUI.

# Сетевые параметры запуска ComfyUI.
COMFY_PORT = int(os.environ.get("COMFY_PORT", "8188"))  # Порт ComfyUI, по умолчанию 8188.
COMFY_LISTEN_HOST_LOCAL = "127.0.0.1"  # Хост для локального режима (доступ только через SSH-туннель).
COMFY_LISTEN_HOST_PUBLIC = "0.0.0.0"  # Хост для публичного режима (нужен для временного сайта).

# Секреты читаются из окружения, но НЕ задаются в ноутбуке (по вашему требованию).
CIVITAI_TOKEN = os.environ.get("CIVITAI_TOKEN", "")  # Токен Civitai.
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # Токен Hugging Face.
GRADIO_TOKEN = os.environ.get("GRADIO_TOKEN", "")  # Токен авторизации временного Gradio-доступа.

# Создаём все нужные директории заранее, чтобы следующие ячейки не падали на FileNotFoundError.
required_directories = [
    VAST_VOLUME_DIR,
    COMFY_ROOT,
    MODELS_DIR,
    CHECKPOINTS_DIR,
    LORAS_DIR,
    VAE_DIR,
    UPSCALE_DIR,
    CONTROLNET_DIR,
    IPADAPTER_DIR,
    CLIP_VISION_DIR,
    CUSTOM_NODES_DIR,
    INPUT_DIR,
    OUTPUT_DIR,
    USER_DIR,
]
for directory_path in required_directories:  # Идём по всем путям из списка.
    directory_path.mkdir(parents=True, exist_ok=True)  # Создаём директорию, если её ещё нет.

# Упаковываем настройки в единый словарь для переиспользования в следующих ячейках.
CONFIG = {
    "VAST_VOLUME_DIR": str(VAST_VOLUME_DIR),
    "COMFY_ROOT": str(COMFY_ROOT),
    "MODELS_DIR": str(MODELS_DIR),
    "CHECKPOINTS_DIR": str(CHECKPOINTS_DIR),
    "LORAS_DIR": str(LORAS_DIR),
    "VAE_DIR": str(VAE_DIR),
    "UPSCALE_DIR": str(UPSCALE_DIR),
    "CONTROLNET_DIR": str(CONTROLNET_DIR),
    "IPADAPTER_DIR": str(IPADAPTER_DIR),
    "CLIP_VISION_DIR": str(CLIP_VISION_DIR),
    "CUSTOM_NODES_DIR": str(CUSTOM_NODES_DIR),
    "INPUT_DIR": str(INPUT_DIR),
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "USER_DIR": str(USER_DIR),
    "COMFY_PORT": COMFY_PORT,
    "COMFY_LISTEN_HOST_LOCAL": COMFY_LISTEN_HOST_LOCAL,
    "COMFY_LISTEN_HOST_PUBLIC": COMFY_LISTEN_HOST_PUBLIC,
    "HAS_CIVITAI_TOKEN": bool(CIVITAI_TOKEN),
    "HAS_HF_TOKEN": bool(HF_TOKEN),
    "HAS_GRADIO_TOKEN": bool(GRADIO_TOKEN),
}

print("Настройки ComfyUI для Vast подготовлены.")
for key_name, value in CONFIG.items():  # Печатаем ключи/значения для прозрачности.
    print(f"- {key_name}: {value}")


In [ ]:
# ЯЧЕЙКА 3: Установка зависимостей и ComfyUI (если ещё не установлены).
# Логика: сначала проверяем наличие, потом устанавливаем только недостающее.

import os  # Модуль для переменных окружения.
import shutil  # Модуль для проверки существования системных утилит.
import subprocess  # Модуль для запуска shell-команд.
import sys  # Модуль для доступа к текущему Python-интерпретатору.
from pathlib import Path  # Удобная работа с путями.

# Если переменная CONFIG уже создана в ячейке 2 — используем её.
# Иначе задаём безопасные значения по умолчанию, чтобы ячейка могла запуститься отдельно.
comfy_root = Path(globals().get("CONFIG", {}).get("COMFY_ROOT", "/workspace/ComfyUI"))  # Корень ComfyUI.
custom_nodes_dir = comfy_root / "custom_nodes"  # Папка custom_nodes.


def run_shell(command_text, step_title):
    # Функция запускает shell-команду и прекращает выполнение при ошибке.
    # command_text: строка shell-команды.
    # step_title: имя шага для понятного лога.
    print(f"
[STEP] {step_title}")
    print(f"[CMD] {command_text}")
    completed_process = subprocess.run(command_text, shell=True, text=True)
    if completed_process.returncode != 0:
        raise RuntimeError(f"Ошибка на шаге: {step_title}")


# Проверяем наличие базовых системных утилит.
required_binaries = ["git"]  # Минимальный список: git нужен для clone/pull.
for binary_name in required_binaries:  # Идём по каждой утилите.
    if shutil.which(binary_name) is None:  # Если утилита не найдена в PATH.
        raise RuntimeError(f"Не найдена системная утилита: {binary_name}")

# Обновляем pip/setuptools/wheel для более стабильной установки Python-пакетов.
run_shell(
    f"{sys.executable} -m pip install -U pip setuptools wheel",
    "Обновление pip-инструментов",
)

# Список python-зависимостей для ComfyUI и скачивающей ячейки.
pip_packages = [
    "aria2p",  # API-клиент для aria2 (ускоренные многопоточные загрузки).
    "huggingface_hub",  # Загрузка файлов с Hugging Face.
    "requests",  # HTTP-клиент для прямых URL-скачиваний.
    "tqdm",  # Прогресс-бары.
    "gradio>=4.0.0",  # Временный веб-доступ с auth во 2-м режиме запуска.
]
run_shell(
    f"{sys.executable} -m pip install -U " + " ".join(pip_packages),
    "Установка python-зависимостей",
)

# Проверяем наличие ComfyUI. Если папка .git есть — делаем git pull, иначе clone.
if (comfy_root / ".git").exists():  # ComfyUI уже клонирован.
    run_shell(f"git -C {comfy_root} pull", "Обновление ComfyUI через git pull")
else:  # ComfyUI отсутствует.
    run_shell(f"git clone https://github.com/comfyanonymous/ComfyUI.git {comfy_root}", "Клонирование ComfyUI")

# Устанавливаем зависимости самого ComfyUI из requirements.txt.
requirements_file = comfy_root / "requirements.txt"  # Файл зависимостей ComfyUI.
if requirements_file.exists():  # Если файл requirements найден.
    run_shell(
        f"{sys.executable} -m pip install -U -r {requirements_file}",
        "Установка зависимостей ComfyUI",
    )
else:  # Если requirements.txt отсутствует.
    raise FileNotFoundError(f"Не найден файл зависимостей: {requirements_file}")

# Гарантируем наличие custom_nodes перед следующей ячейкой.
custom_nodes_dir.mkdir(parents=True, exist_ok=True)  # Создаём папку custom_nodes при необходимости.

print("
Готово: зависимости и база ComfyUI подготовлены.")


In [ ]:
# ЯЧЕЙКА 4: Формирование списков загрузки + параллельная загрузка (до 5 потоков) с приоритетом.
# Логика по ТЗ: сначала формируем списки, затем качаем параллельно.
# Приоритет: сначала тяжёлые файлы, затем лёгкие.

import os  # Для чтения токенов окружения.
import subprocess  # Для git clone/pull.
from concurrent.futures import ThreadPoolExecutor, as_completed  # Для параллельной загрузки.
from pathlib import Path  # Для безопасной работы с путями.

import requests  # Для скачивания файлов по прямым URL.
from huggingface_hub import hf_hub_download  # Для загрузки файлов с Hugging Face.

# Подхватываем конфиг из ячейки 2.
cfg = globals().get("CONFIG", {})  # Словарь настроек, созданный ранее.
comfy_root = Path(cfg.get("COMFY_ROOT", "/workspace/ComfyUI"))  # Корень ComfyUI.
custom_nodes_dir = Path(cfg.get("CUSTOM_NODES_DIR", "/workspace/ComfyUI/custom_nodes"))  # Папка custom_nodes.
controlnet_dir = Path(cfg.get("CONTROLNET_DIR", "/workspace/ComfyUI/models/controlnet"))  # Папка controlnet.
ipadapter_dir = Path(cfg.get("IPADAPTER_DIR", "/workspace/ComfyUI/models/ipadapter"))  # Папка ipadapter.

hf_token = os.environ.get("HF_TOKEN", "")  # Токен Hugging Face из окружения.


def ensure_git_repo(repo_url, destination_dir):
    # Функция гарантирует, что git-репозиторий существует и актуален.
    # repo_url: URL репозитория GitHub.
    # destination_dir: локальная папка, куда клонируем репозиторий.
    destination_dir = Path(destination_dir)
    if (destination_dir / ".git").exists():  # Если репозиторий уже есть.
        command = ["git", "-C", str(destination_dir), "pull"]  # Команда обновления.
        action_text = "git pull"
    else:  # Если репозитория нет.
        destination_dir.parent.mkdir(parents=True, exist_ok=True)  # Создаём родительскую папку.
        command = ["git", "clone", repo_url, str(destination_dir)]  # Команда первичного clone.
        action_text = "git clone"

    print(f"[REPO] {action_text}: {repo_url}")
    result = subprocess.run(command, text=True, capture_output=True)  # Выполняем git-команду.
    if result.returncode != 0:  # Если git вернул ошибку.
        raise RuntimeError(f"Ошибка git для {repo_url}: {result.stderr}")


def download_http_file(url, target_path):
    # Функция скачивает один файл по HTTP/HTTPS URL.
    # url: источник файла.
    # target_path: локальный путь, куда сохранить файл.
    target_path = Path(target_path)
    if target_path.exists():  # Если файл уже есть.
        return f"SKIP (exists): {target_path.name}"

    target_path.parent.mkdir(parents=True, exist_ok=True)  # Создаём папку назначения.
    with requests.get(url, stream=True, timeout=120) as response:  # Открываем поток загрузки.
        response.raise_for_status()  # Проверяем HTTP-код ответа.
        with open(target_path, "wb") as file_object:  # Открываем локальный файл на запись.
            for chunk_bytes in response.iter_content(chunk_size=1024 * 1024):  # Читаем чанками по 1MB.
                if chunk_bytes:  # Защита от пустых чанков keep-alive.
                    file_object.write(chunk_bytes)  # Записываем кусок в файл.
    return f"OK: {target_path.name}"


def download_hf_file(repo_id, filename, target_dir, token):
    # Функция скачивает файл с Hugging Face Hub и кладёт его в целевую папку.
    # repo_id: имя HF-репозитория (например 'h94/IP-Adapter-FaceID').
    # filename: имя файла внутри репозитория.
    # target_dir: папка назначения в ComfyUI.
    # token: HF-токен (может быть пустым для публичных файлов).
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    target_file = target_dir / filename

    if target_file.exists():  # Если файл уже есть.
        return f"SKIP (exists): {filename}"

    downloaded_path = hf_hub_download(  # Скачиваем файл в кэш HF.
        repo_id=repo_id,
        filename=filename,
        token=token if token else None,
    )
    Path(downloaded_path).replace(target_file)  # Перемещаем файл из кэша в нужную папку.
    return f"OK: {filename}"


# 1) Список custom nodes (обязательно по ТЗ).
custom_node_repos = [
    "https://github.com/ltdrdata/ComfyUI-Manager.git",  # Менеджер расширений ComfyUI.
    "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",  # IP-Adapter FaceID PlusV2 ноды.
    "https://github.com/Fannovel16/comfyui_controlnet_aux.git",  # Препроцессоры ControlNet.
]

# 2) Список тяжёлых файлов (высокий приоритет).
heavy_download_jobs = [
    {
        "name": "ip-adapter-faceid-plusv2_sdxl.bin",
        "kind": "hf",
        "repo_id": "h94/IP-Adapter-FaceID",
        "filename": "ip-adapter-faceid-plusv2_sdxl.bin",
        "target_dir": str(ipadapter_dir),
    },
]

# 3) Список лёгких/дополнительных файлов (можно расширять).
light_download_jobs = [
    # Пример заготовки под future download:
    # {
    #     "name": "example_model.safetensors",
    #     "kind": "http",
    #     "url": "https://example.com/example_model.safetensors",
    #     "target_path": str(controlnet_dir / "example_model.safetensors"),
    # },
]

# Сначала устанавливаем/обновляем custom nodes.
for repo_url in custom_node_repos:  # Проходим по каждому репозиторию расширений.
    repo_name = Path(repo_url).stem  # Имя папки из URL репозитория.
    destination = custom_nodes_dir / repo_name  # Путь установки конкретного custom node.
    ensure_git_repo(repo_url, destination)  # Выполняем clone/pull.

# Затем формируем общий список задач в порядке приоритета: heavy -> light.
ordered_jobs = heavy_download_jobs + light_download_jobs  # Итоговый порядок задач.


def execute_download_job(job_dict):
    # Функция выполняет одну задачу скачивания из словаря job_dict.
    # job_dict: словарь с типом задачи и параметрами загрузки.
    job_kind = job_dict["kind"]  # Тип источника: 'hf' или 'http'.
    if job_kind == "hf":  # Загрузка с Hugging Face.
        return download_hf_file(
            repo_id=job_dict["repo_id"],
            filename=job_dict["filename"],
            target_dir=job_dict["target_dir"],
            token=hf_token,
        )
    if job_kind == "http":  # Загрузка по прямому URL.
        return download_http_file(
            url=job_dict["url"],
            target_path=job_dict["target_path"],
        )
    raise ValueError(f"Неизвестный тип загрузки: {job_kind}")  # Ошибка на неподдерживаемом типе.


# Параллельная загрузка: максимум 5 потоков по вашему требованию.
max_workers_count = 5  # Максимум одновременных загрузок.
if ordered_jobs:  # Если список задач не пустой.
    with ThreadPoolExecutor(max_workers=max_workers_count) as pool:  # Создаём пул потоков.
        future_to_job = {pool.submit(execute_download_job, job): job for job in ordered_jobs}  # Планируем все задачи.
        for future in as_completed(future_to_job):  # Обрабатываем завершение задач по мере готовности.
            job_info = future_to_job[future]  # Получаем описание завершённой задачи.
            try:  # Пытаемся получить результат задачи.
                message = future.result()  # Текст успеха/пропуска.
                print(f"[DOWNLOAD] {job_info['name']}: {message}")
            except Exception as error_object:  # Ловим ошибку конкретной задачи.
                print(f"[DOWNLOAD][ERROR] {job_info['name']}: {error_object}")
else:  # Если задач нет.
    print("Список загрузок пуст. Нечего скачивать.")

print("Готово: custom nodes и модели обработаны.")


In [ ]:
# ЯЧЕЙКА 5: Запуск ComfyUI в двух режимах по выбору пользователя.
# Режим 1: локальный запуск (для SSH-туннеля на Vast, без внешнего доступа).
# Режим 2: временный веб-доступ через Gradio с авторизацией (экстренный режим/Colab-тест).

import os  # Для чтения переменных окружения.
import socket  # Для проверки доступности локального порта ComfyUI.
import subprocess  # Для запуска ComfyUI как фонового процесса.
import sys  # Для пути к python-интерпретатору.
import time  # Для пауз между проверками старта.
from pathlib import Path  # Для путей к файлам проекта.

import gradio as gr  # Для создания временного сайта-прокси с auth.
import requests  # Для HTTP-проксирования запросов к локальному ComfyUI.

# Загружаем настройки из CONFIG (создан в ячейке 2).
cfg = globals().get("CONFIG", {})  # Словарь конфигурации.
comfy_root = Path(cfg.get("COMFY_ROOT", "/workspace/ComfyUI"))  # Папка ComfyUI.
comfy_port = int(cfg.get("COMFY_PORT", 8188))  # Порт ComfyUI.
local_host = cfg.get("COMFY_LISTEN_HOST_LOCAL", "127.0.0.1")  # Хост для локального режима.
public_host = cfg.get("COMFY_LISTEN_HOST_PUBLIC", "0.0.0.0")  # Хост для публичного режима.
gradio_token = os.environ.get("GRADIO_TOKEN", "")  # Токен auth для Gradio-режима.


def is_port_open(host, port):
    # Функция проверяет, слушает ли процесс указанный host:port.
    # host: IP/домен для проверки.
    # port: числовой TCP-порт.
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:  # Создаём TCP-сокет.
        sock.settimeout(1.5)  # Ограничиваем время ожидания подключения.
        return sock.connect_ex((host, port)) == 0  # True, если порт открыт.


def start_comfy_process(listen_host):
    # Функция запускает ComfyUI в фоне.
    # listen_host: адрес bind для параметра --listen.
    main_py = comfy_root / "main.py"  # Главный исполняемый файл ComfyUI.
    if not main_py.exists():  # Проверка наличия main.py.
        raise FileNotFoundError(f"Не найден файл запуска ComfyUI: {main_py}")

    command_list = [  # Команда запуска ComfyUI.
        sys.executable,
        str(main_py),
        "--listen",
        listen_host,
        "--port",
        str(comfy_port),
    ]
    print("[CMD]", " ".join(command_list))

    # Запускаем процесс в фоне и сохраняем PID в переменную окружения.
    process = subprocess.Popen(command_list, cwd=str(comfy_root))
    os.environ["COMFYUI_PID"] = str(process.pid)  # PID пригодится для ячейки остановки.
    return process.pid  # Возвращаем PID для лога.


def wait_comfy_ready(host, port, timeout_sec=120):
    # Функция ждёт, пока ComfyUI начнёт отвечать на указанном порту.
    # host: адрес подключения.
    # port: порт подключения.
    # timeout_sec: максимум секунд ожидания.
    start_time = time.time()  # Время старта ожидания.
    while time.time() - start_time < timeout_sec:  # Цикл до истечения таймаута.
        if is_port_open(host, port):  # Если порт уже слушается.
            return True
        time.sleep(1)  # Пауза 1 сек между попытками.
    return False  # Таймаут истёк.


print("1) Вариант запуска: локальный режим (SSH-туннель, без внешнего доступа)")
print("2) Вариант запуска: временный сайт через Gradio + auth")
launch_mode = input("Введите режим (1 или 2): ").strip()  # Пользовательский выбор режима.

if launch_mode not in {"1", "2"}:  # Валидация ввода.
    raise ValueError("Нужно ввести только 1 или 2.")

if launch_mode == "1":  # Локальный режим.
    comfy_pid = start_comfy_process(local_host)  # Запускаем ComfyUI на 127.0.0.1.
    ready = wait_comfy_ready(local_host, comfy_port)  # Ждём готовность сервиса.
    if not ready:
        raise RuntimeError("ComfyUI не поднялся в локальном режиме за отведённое время.")
    print(f"ComfyUI запущен (PID={comfy_pid}) на http://{local_host}:{comfy_port}")
    print("Для доступа с локальной машины используйте SSH-туннель Vast.ai.")

if launch_mode == "2":  # Режим временного внешнего доступа.
    comfy_pid = start_comfy_process(public_host)  # Запускаем ComfyUI на 0.0.0.0.
    ready = wait_comfy_ready("127.0.0.1", comfy_port)  # Проверяем локально внутри инстанса.
    if not ready:
        raise RuntimeError("ComfyUI не поднялся в режиме временного сайта за отведённое время.")

    # Проверяем, что задан токен для basic-auth защиты Gradio.
    if not gradio_token:
        raise RuntimeError("GRADIO_TOKEN не задан. Для режима 2 нужен токен авторизации.")

    def proxy_get(path):
        # Функция-прокси для GET-запроса к локальному ComfyUI.
        # path: относительный путь endpoint (например '/queue').
        target_url = f"http://127.0.0.1:{comfy_port}{path}"  # Целевой URL ComfyUI.
        response = requests.get(target_url, timeout=30)  # Выполняем запрос к локальному сервису.
        return response.text  # Возвращаем текст ответа.

    with gr.Blocks() as demo:  # Создаём простой Gradio-интерфейс-прокси.
        gr.Markdown("# ComfyUI emergency access proxy")  # Заголовок страницы.
        endpoint_input = gr.Textbox(value="/", label="Endpoint path")  # Поле для пути endpoint.
        result_output = gr.Textbox(label="Response text")  # Поле для ответа от ComfyUI.
        send_button = gr.Button("GET")  # Кнопка отправки запроса.
        send_button.click(fn=proxy_get, inputs=endpoint_input, outputs=result_output)  # Связываем кнопку и функцию.

    demo.launch(
        share=True,  # Создаём временную публичную ссылку Gradio.
        auth=("user", gradio_token),  # Включаем auth с логином user и паролем из GRADIO_TOKEN.
        server_name="0.0.0.0",  # Разрешаем внешний доступ к Gradio внутри инстанса.
    )


In [ ]:
# ЯЧЕЙКА 6: Остановка (убийство) процесса ComfyUI.
# По вашему ТЗ эта ячейка заменяет старую неактуальную функциональность.

import os  # Для чтения сохранённого PID из переменной окружения.
import signal  # Для отправки сигнала завершения процессу.
import subprocess  # Для поиска PID, если переменная окружения не заполнена.

saved_pid_text = os.environ.get("COMFYUI_PID", "")  # PID ComfyUI, сохранённый в ячейке запуска.

if saved_pid_text:  # Если PID найден в окружении.
    comfy_pid = int(saved_pid_text)  # Преобразуем строку PID в число.
    try:  # Пытаемся аккуратно остановить процесс.
        os.kill(comfy_pid, signal.SIGTERM)  # Отправляем сигнал завершения SIGTERM.
        print(f"ComfyUI остановлен по PID={comfy_pid}")
    except ProcessLookupError:  # Если процесс с таким PID уже не существует.
        print(f"Процесс PID={comfy_pid} уже завершён.")
else:  # Если PID не сохранён в окружении.
    # Пытаемся найти возможные процессы ComfyUI через ps и завершаем их.
    ps_command = "ps -eo pid,cmd | grep 'ComfyUI/main.py' | grep -v grep"  # Команда поиска запущенного ComfyUI.
    result = subprocess.run(ps_command, shell=True, text=True, capture_output=True)  # Выполняем поиск.
    if result.returncode != 0 or not result.stdout.strip():  # Если процессов не найдено.
        print("Запущенный процесс ComfyUI не найден.")
    else:  # Если найден хотя бы один процесс.
        for line in result.stdout.strip().splitlines():  # Обрабатываем каждую найденную строку.
            pid_value = int(line.strip().split()[0])  # Извлекаем PID из строки ps.
            os.kill(pid_value, signal.SIGTERM)  # Останавливаем найденный процесс.
            print(f"ComfyUI остановлен по PID={pid_value}")
